# Drivers Graphs

This notebook is made to fetch driver telemetry data in a single track in a single year

In [68]:
import fastf1
import os

YEAR = 2024
DRIVER_ABV = "VER"
RACE = "Monaco"

session = fastf1.get_session(YEAR, RACE, "R")
session.load(laps=True, telemetry=True)
driver_laps = session.laps.pick_drivers(DRIVER_ABV)
fastest_lap = driver_laps.pick_fastest()
start_td = fastest_lap["Time"] - fastest_lap["LapTime"]
end_td = fastest_lap["Time"]
telemetry = fastest_lap.get_telemetry()
telemetry['RelativeSeconds'] = (telemetry['SessionTime'] - start_td).dt.total_seconds()

position_points = []
for row in telemetry.itertuples():
    position_points.append((
        round(row.RelativeSeconds, 4), 
        row.X, 
        row.Y,
        row.Z,
        row.RPM, 
        row.Speed, 
        row.nGear, 
        row.Throttle, 
        row.Brake, 
        row.DRS, 
        row.Status
    ))

os.makedirs('data', exist_ok=True)
filename = f'data/telemetry_{DRIVER_ABV}_{RACE}_{YEAR}.csv'

with open(filename, 'w') as f:
    f.write("seconds,x,y,z,rpm,speed,gear,throttle,brake,drs,status\n")
    for point in position_points:
        f.write(','.join(map(str, point)) + '\n')

print(f"Exported {len(position_points)} points for {DRIVER_ABV} in {RACE}")

core           INFO 	Loading data for Monaco Grand Prix - Race [v3.8.1]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for lap_count
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...
req            INFO 	Using cached data for car_data
req            INFO 	Using cached data for position_data
req            INFO 	Using cached data for weather_data
req            INFO 	Using cached data for race_control_messages
core           INFO 	Finished loading data for 20 drivers: ['16', '81', '55', '4', '63', '1', '44', '22', '23', '10', '14', '3', '77', '18', '2', '24', '31', '11', '27', '20']


Exported 573 points for VER in Monaco
